# Estudio de tres años con entradas de un minuto

**Entregable 1 — Jassan Arteta y Mateo Bernal**  
Universidad del Norte · Maestría en Matemáticas · Machine Learning · Profesor Lihki Rubio · 2026-30

Periodo: **1 de enero de 2023 a 31 de diciembre de 2025 (UTC)**. BTCUSDT, ETHUSDT, BNBUSDT y XRPUSDT.
Entrenamiento 2023; validación 2024; evaluación retrospectiva 2025. Esta última ya se ha examinado en experimentos previos.

Las entradas conservan cada cierre de un minuto: 10.080, 20.160, 30.240 o 40.320 valores para ventanas de 7, 14, 21 o 28 días.
Se emite un pronóstico al cierre de cada día elegible. Las siete salidas son volatilidades diarias futuras:
100 veces la desviación estándar poblacional de retornos logarítmicos de cierres diarios, en ventanas de 7, 14, 21 o 28 días.
No se anualiza. Los días con minutos faltantes se excluyen cuando afectan una entrada o un objetivo; no se interpolan.
En este conjunto, el día incompleto es el 24 de marzo de 2023. Todas las ventanas comparten el calendario elegible
definido por la historia máxima, para que su comparación no cambie de fechas según el tamaño de entrada.

El escalado se ajusta exclusivamente en entrenamiento. LinearSVR se resuelve en una base ortonormal del espacio generado
por las filas de entrenamiento, conservando todas las direcciones numéricamente no nulas. Se verifica la reconstrucción
de la matriz de productos internos y se exportan coeficientes para todos los minutos originales.
No se sustituyen las entradas por cierres diarios. C, epsilon y ventana de entrada se eligen por RMSE de validación,
separadamente para cada cripto y definición de volatilidad.

La ejecución paralela reutilizó búsquedas ya terminadas, conservadas en selection_cache.csv, y volvió a verificar sus errores de validación. Para esas configuraciones, search.csv contiene la verificación del candidato seleccionado; para las nuevas, contiene la búsqueda completa. Una ejecución desde cero, sin caché, evalúa todos los candidatos.

## Validación

Se ejecutan GroupKFold, KFold y ForwardChaining de timeseries-cv con filtros cronológicos.
Los adaptadores de KFold y ForwardChaining agrupan cortes nativos y, tras filtrar, generan exactamente las mismas muestras.
Sus cinco particiones de evaluación comparten entrenamiento: no son cinco reentrenamientos independientes.
GroupKFold conserva pocas muestras. El archivo all_metrics.csv distingue fechas comunes de todas las fechas disponibles.
Las diferencias de rendimiento entre GroupKFold y los otros adaptadores combinan el método de partición
y la cantidad de entrenamiento retenida; GroupKFold conserva entre 9 y 12 muestras de entrenamiento por fold.
Comparar métodos exige usar common_test; los resultados siguientes usan todas las fechas de ForwardChaining.
KFold y ForwardChaining usan 294 orígenes únicos de entrenamiento, 359 de validación
y 358 de prueba. Hay 21 fechas de prueba comunes a los tres métodos.
Los millones de cierres de minuto constituyen las columnas de las ventanas; no equivalen a millones de muestras supervisadas.

## Exploración

descriptive_statistics.csv contiene estadísticas de cierres y retornos de un minuto y de retornos diarios.
Las figuras usan cierres diarios para legibilidad y la frecuencia diaria del objetivo para los diagnósticos.
Los huecos se mantienen; la ACF usa tratamiento conservador de faltantes. La ACF de cuadrados permite inspeccionar
dependencia en la magnitud de los retornos, sin demostrar por sí sola un modelo de heterocedasticidad.

- **BTCUSDT:** 1,578,159 cierres de minuto válidos y 81 faltantes. Los retornos logarítmicos diarios tienen desviación estándar de 2.42 %, mínimo -8.93 % y máximo 11.23 %. Los extremos justifican comparar errores cuadrados con MAE, menos sensible a valores extremos.
- **ETHUSDT:** 1,578,159 cierres de minuto válidos y 81 faltantes. Los retornos logarítmicos diarios tienen desviación estándar de 3.28 %, mínimo -15.85 % y máximo 19.79 %. Los extremos justifican comparar errores cuadrados con MAE, menos sensible a valores extremos.
- **BNBUSDT:** 1,578,159 cierres de minuto válidos y 81 faltantes. Los retornos logarítmicos diarios tienen desviación estándar de 2.71 %, mínimo -12.99 % y máximo 15.96 %. Los extremos justifican comparar errores cuadrados con MAE, menos sensible a valores extremos.
- **XRPUSDT:** 1,578,159 cierres de minuto válidos y 81 faltantes. Los retornos logarítmicos diarios tienen desviación estándar de 4.25 %, mínimo -20.81 % y máximo 54.87 %. Los extremos justifican comparar errores cuadrados con MAE, menos sensible a valores extremos.

![EDA BTCUSDT](../book/figures/minute_eda_BTCUSDT.png)

![EDA ETHUSDT](../book/figures/minute_eda_ETHUSDT.png)

![EDA BNBUSDT](../book/figures/minute_eda_BNBUSDT.png)

![EDA XRPUSDT](../book/figures/minute_eda_XRPUSDT.png)

## Resultados

Se evaluaron 192 combinaciones método–cripto–entrada–objetivo.
SVR supera a persistencia en RMSE en **0 de 16** selecciones del método principal.
Persistencia repite la última volatilidad conocida en los siete horizontes.
Se conservan 1900 predicciones negativas del SVR entre 40096 valores pronosticados seleccionados.
No se aplica recorte antes de evaluar; una predicción negativa no representa una volatilidad válida.
R² se promedia entre horizontes; por cripto se promedia entre cuatro objetivos, y GLOBAL_MACRO entre las 16 selecciones.
El global es un promedio macro, no un R² obtenido concatenando escalas distintas.
RMSE también promedia los siete RMSE por horizonte; no es la raíz de un único MSE global.
RMSE y MAE se expresan en puntos porcentuales de volatilidad, MSE en su cuadrado y MAPE en porcentaje.

| symbol | model | r2 | rmse | mae | mse | mape |
| --- | --- | --- | --- | --- | --- | --- |
| BNBUSDT | Persistence | 0.67199 | 0.56963 | 0.40284 | 0.43591 | 19.38251 |
| BNBUSDT | SVR | -4.72175 | 2.54052 | 2.248 | 6.59008 | 118.97934 |
| BTCUSDT | Persistence | 0.59362 | 0.43642 | 0.3069 | 0.25727 | 16.96313 |
| BTCUSDT | SVR | -2.76504 | 1.39495 | 1.05771 | 2.07383 | 51.82509 |
| ETHUSDT | Persistence | 0.34257 | 0.89615 | 0.61315 | 1.0904 | 19.56662 |
| ETHUSDT | SVR | -1.70908 | 1.77232 | 1.32482 | 3.20626 | 34.50461 |
| XRPUSDT | Persistence | 0.54275 | 1.0886 | 0.66589 | 1.5292 | 19.31856 |
| XRPUSDT | SVR | -55.13094 | 10.39844 | 9.24536 | 129.2181 | 268.33068 |
| GLOBAL_MACRO | Persistence | 0.53773 | 0.7477 | 0.49719 | 0.8282 | 18.80771 |
| GLOBAL_MACRO | SVR | -16.0817 | 4.02656 | 3.46897 | 35.27207 | 118.40993 |

![R² y RMSE](../book/figures/minute_2023_2025.png)

Reducir años y cambiar de cierres diarios a entradas de un minuto modifica simultáneamente muestra y representación.
Por ello no se atribuye una diferencia de métricas exclusivamente a reducir el periodo. Más predictores que observaciones
de entrenamiento puede perjudicar la generalización. Los objetivos de volatilidad móvil comparten retornos conocidos,
lo que favorece un baseline persistente; superar ese baseline sigue siendo la referencia pertinente.

El archivo bds.csv conserva el diagnóstico BDS de residuos del primer horizonte por partición.
GroupKFold tiene menos de seis observaciones por partición y se registra como no estimable.
En los demás métodos, las particiones contienen fechas espaciadas: este diagnóstico es exploratorio
y no equivale a una prueba sobre todos los residuos diarios consecutivos.

Para revisar generalización, selected_split_metrics.csv evalúa los mismos 16 modelos seleccionados en cada periodo,
sin reentrenarlos. Su R² macro de entrenamiento es 0.3650,
frente a -1.4080 en validación y -16.0817 en prueba.
Una brecha grande señala que el ajuste dentro de 2023 no se traslada a años posteriores; acortar el periodo
no resuelve por sí solo la generalización.

## Reproducción

Configuración: experiment_minute.json. Ejecutar, desde la raíz y con dependencias instaladas:

```bash
python src/run_minute_experiment.py
python src/report_minute_experiment.py
python src/verify_minute_experiment.py
```

La entrega incluye los datos procesados de un minuto. Para reconstruirlos desde archivos mensuales de Binance
ya descargados en data/raw/minute_2020_2025, ejecutar python src/minute_experiment.py.
Si faltan los originales, python src/download_three_years.py descarga únicamente los 144 archivos necesarios.
Resultados, calendario, métricas por horizonte y modelos: results/minute_2023_2025.
El estudio anterior con entradas diarias permanece como referencia histórica.


In [1]:
from pathlib import Path
import pandas as pd
ROOT=Path.cwd()
if not (ROOT/'experiment_minute.json').exists(): ROOT=ROOT.parent
OUT=ROOT/'results/minute_2023_2025'
assert OUT.exists()


La siguiente celda permite repetir el entrenamiento completo usando los datos de un minuto incluidos. Por defecto se consultan los resultados ejecutados. El entrenamiento puede tardar decenas de minutos.

In [2]:
RETRAIN=False
if RETRAIN:
    import subprocess, sys
    subprocess.run([sys.executable,str(ROOT/'src/run_minute_experiment.py')],cwd=ROOT,check=True)

In [3]:
pd.read_csv(OUT/'descriptive_statistics.csv')

,symbol,series,n,missing,mean,std,min,median,max
0,BTCUSDT,minute_close,1578159,81,6.544668e+04,31818.356763,16505.870000,64212.130000,126114.500000
1,BTCUSDT,minute_log_return,1578157,82,1.059997e-06,0.000659,-0.043845,0.000000,0.031066
2,BTCUSDT,daily_log_return,1093,3,1.548748e-03,0.024209,-0.089254,0.000316,0.112287
3,ETHUSDT,minute_close,1578159,81,2.635080e+03,841.306783,1191.540000,2521.300000,4954.640000
4,ETHUSDT,minute_log_return,1578157,82,5.858491e-07,0.000873,-0.100804,0.000000,0.047660
5,ETHUSDT,daily_log_return,1093,3,8.663886e-04,0.032847,-0.158501,0.000589,0.197871
6,BNBUSDT,minute_close,1578159,81,5.270031e+02,238.944721,202.700000,573.400000,1373.920000
7,BNBUSDT,minute_log_return,1578157,82,7.989979e-07,0.000797,-0.083036,0.000000,0.072552
8,BNBUSDT,daily_log_return,1093,3,1.175291e-03,0.027073,-0.129861,0.001623,0.159584
9,XRPUSDT,minute_close,1578159,81,1.246205e+00,0.965318,0.316700,0.606700,3.653700


In [4]:
pd.read_csv(OUT/'macro_metrics.csv').query("scope == 'all_available_test' and method == 'ForwardChaining'")

,scope,method,symbol,model,r2,rmse,mae,mse,mape
0,all_available_test,ForwardChaining,BNBUSDT,Persistence,0.671986,0.569631,0.402837,0.435908,19.382514
1,all_available_test,ForwardChaining,BNBUSDT,SVR,-4.721745,2.540516,2.248002,6.590077,118.979342
2,all_available_test,ForwardChaining,BTCUSDT,Persistence,0.593623,0.436421,0.306897,0.257269,16.963125
3,all_available_test,ForwardChaining,BTCUSDT,SVR,-2.765042,1.394952,1.057705,2.073830,51.825093
4,all_available_test,ForwardChaining,ETHUSDT,Persistence,0.342572,0.896152,0.613152,1.090404,19.566617
5,all_available_test,ForwardChaining,ETHUSDT,SVR,-1.709080,1.772321,1.324818,3.206257,34.504609
6,all_available_test,ForwardChaining,XRPUSDT,Persistence,0.542750,1.088597,0.665889,1.529205,19.318564
7,all_available_test,ForwardChaining,XRPUSDT,SVR,-55.130938,10.398443,9.245360,129.218097,268.330681
48,all_available_test,ForwardChaining,GLOBAL_MACRO,Persistence,0.537732,0.747700,0.497194,0.828196,18.807705
49,all_available_test,ForwardChaining,GLOBAL_MACRO,SVR,-16.081701,4.026558,3.468971,35.272065,118.409931


In [5]:
pd.read_csv(OUT/'selected_inputs.csv')

,method,symbol,volatility_window,input_window,C,epsilon,val_rmse,n_features,selection_source
0,ForwardChaining,BNBUSDT,7,7,0.01,0.01,1.830609,10080,prior_completed_grid
1,ForwardChaining,BNBUSDT,14,7,0.01,0.10,1.539153,10080,prior_completed_grid
2,ForwardChaining,BNBUSDT,21,7,0.01,0.10,1.423667,10080,prior_completed_grid
3,ForwardChaining,BNBUSDT,28,7,0.01,0.10,1.304214,10080,prior_completed_grid
4,ForwardChaining,BTCUSDT,7,7,0.01,0.01,1.608182,10080,prior_completed_grid
5,ForwardChaining,BTCUSDT,14,7,0.01,0.01,1.330222,10080,prior_completed_grid
6,ForwardChaining,BTCUSDT,21,7,0.01,0.01,1.228610,10080,prior_completed_grid
7,ForwardChaining,BTCUSDT,28,7,0.01,0.01,1.132803,10080,prior_completed_grid
8,ForwardChaining,ETHUSDT,7,7,0.01,0.01,1.594936,10080,prior_completed_grid
9,ForwardChaining,ETHUSDT,14,7,0.01,0.01,1.367861,10080,prior_completed_grid


In [6]:
pd.read_csv(OUT/'selected_split_metrics.csv').groupby(['split','model'])[['r2','rmse','mae']].mean()

r2      rmse       mae
split model                                     
test  Persistence   0.537732  0.747700  0.497194
      SVR         -16.081701  4.026558  3.468971
train Persistence   0.470636  0.883190  0.445339
      SVR           0.365045  0.863381  0.599622
val   Persistence   0.579511  0.696586  0.471046
      SVR          -1.408029  1.835916  1.251971

In [7]:
pd.read_csv(OUT/'macro_metrics.csv').query("scope == 'common_test'")

,scope,method,symbol,model,r2,rmse,mae,mse,mape
24,common_test,ForwardChaining,BNBUSDT,Persistence,0.482116,0.553389,0.413487,0.379586,17.965071
25,common_test,ForwardChaining,BNBUSDT,SVR,-7.172846,2.530968,2.201745,6.570893,103.802840
26,common_test,ForwardChaining,BTCUSDT,Persistence,0.651528,0.295345,0.207074,0.110725,14.236984
27,common_test,ForwardChaining,BTCUSDT,SVR,-2.662770,1.020404,0.877258,1.145911,55.352647
28,common_test,ForwardChaining,ETHUSDT,Persistence,0.339013,0.977583,0.576526,1.186379,16.416372
29,common_test,ForwardChaining,ETHUSDT,SVR,-1.610052,1.898770,1.507078,3.974687,37.607769
30,common_test,ForwardChaining,XRPUSDT,Persistence,0.368213,0.789784,0.610723,0.792230,18.199687
31,common_test,ForwardChaining,XRPUSDT,SVR,-125.612996,9.149517,8.238600,102.952059,242.993083
32,common_test,GroupKFold,BNBUSDT,Persistence,0.482116,0.553389,0.413487,0.379586,17.965071
33,common_test,GroupKFold,BNBUSDT,SVR,-119.971095,10.153249,6.777941,127.389362,329.307676


In [8]:
pd.read_csv(OUT/'calendar.csv').groupby(['method','fold','split']).size().unstack()

split                 test  train  val
method          fold                  
ForwardChaining 1       71    294   72
                2       71    294   72
                3       72    294   72
                4       72    294   71
                5       72    294   72
GroupKFold      1        4     12    4
                2        4     11    4
                3        4     10    4
                4        4      9    4
                5        5      9    5
KFold           1       71    294   72
                2       71    294   72
                3       72    294   72
                4       72    294   71
                5       72    294   72